# 01 — Ingestion, Contracts and Time Alignment

Real telemetry sources disagree about names, units, frequencies and timestamps. The first production problem is therefore not model selection. It is **making observations comparable**.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
import sys
sys.path.insert(0, str(ROOT / 'src'))

In [2]:
from apexsim.config import load_config
from apexsim.data.synthetic import generate_synthetic_sessions
from apexsim.data.validate import validate_canonical_frame

config = load_config(ROOT/'configs/fast.yaml')
path = ROOT/'data/raw/notebook_synthetic.csv'
frame = generate_synthetic_sessions(config, path)
report = validate_canonical_frame(frame)
report.to_dict()

{'rows': 8400,
 'sessions': 8,
 'drivers': 5,
 'tracks': 3,
 'null_cells': 0,
 'duplicate_frames': 0,
 'out_of_range_cells': 0,
 'passed': True}

## Read one row as a sentence

A row does **not** mean “a random collection of columns.” It means:

> At this timestamp, in this session, this driver was at this track position, moving with this state, applying these controls, under these external conditions.

The row becomes meaningful only because every adapter promises the same contract.

In [3]:
frame.head(3).T

,0,1,2
session_id,SYN_000,SYN_000,SYN_000
source,synthetic,synthetic,synthetic
track_id,synthetic_track_0,synthetic_track_0,synthetic_track_0
driver_id,DRV_00,DRV_00,DRV_00
timestamp_s,0.0,0.2,0.4
lap_number,1,1,1
speed_mps,53.631356,53.843576,53.233645
acceleration_mps2,9.377836,1.061101,-3.049659
track_progress_sin,0.0,0.01296,0.02597
track_progress_cos,1.0,0.999916,0.999663


## Source adapters

- **FastF1** gives convenient session/lap/telemetry Pandas objects and caching.
- **OpenF1** gives independent HTTP streams such as car data, location, laps and weather. Those streams must be joined by time, not row number.
- **Synthetic** data gives a controlled offline world for testing and learning.
- **F1 25 UDP** will later provide high-rate packets with richer motion, telemetry, status, damage and setup information.

All adapters end at the same canonical frame.

In [4]:
numeric = frame.select_dtypes('number')
print('Rows:', len(frame))
print('Sessions:', frame.session_id.nunique())
print('Median frame spacing:', frame.groupby('session_id').timestamp_s.diff().median())
print('Null cells:', int(frame.isna().sum().sum()))

Rows: 8400
Sessions: 8
Median frame spacing: 0.20000000000000284
Null cells: 0


## Break-it exercise
Change one throttle value to `1.5`, run validation, and read the failure. Then restore it. Production quality gates should make impossible states loud before training.

In [5]:
broken = frame.copy()
broken.loc[0, 'throttle'] = 1.5
try:
    validate_canonical_frame(broken)
except ValueError as exc:
    print(exc)

Canonical data contract failed: {'rows': 8400, 'sessions': 8, 'drivers': 5, 'tracks': 3, 'null_cells': 0, 'duplicate_frames': 0, 'out_of_range_cells': 1, 'passed': False}
